In [0]:
dbutils.widgets.text("environment_name", "")

In [0]:
env_name = dbutils.widgets.get("environment_name")
print(f"environment name is {env_name}")

In [0]:
from pyspark.sql.functions import col, trim, upper, initcap

bronze_df = spark.table(f"{env_name}_bronze.orders")

orders_silver_df = (
    bronze_df
        .filter(col("o_orderkey").isNotNull())
        .filter(col("o_totalprice") > 0)
        .withColumn("o_orderstatus", initcap(trim(col("o_orderstatus"))))
        .dropDuplicates(["o_orderkey"])
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env_name}_silver")

(
    orders_silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{env_name}_silver.orders")
)

In [0]:
display(spark.sql(f"SELECT * FROM {env_name}_silver.orders LIMIT 20"))